In [1]:
from dotenv import load_dotenv
load_dotenv("../.env")

True

In [12]:
import json
from uuid import uuid4
from pathlib import Path

import pandas as pd

from rich import print
from openai import OpenAI
from pydantic import BaseModel
from beir.datasets.data_loader import GenericDataLoader

In [5]:
df_corpus = pd.read_json(f"../data/corpus.jsonl", lines=True)
df_corpus.head()

,id,title,content,published_at,word_count,source_url
0,0f438470-de7f-47cb-8ba2-16e8b1ff5750,JEMBAR SABAR,bismillah yuga lampah balukar janglar meunang ...,2023-07-24,66,https://basasunda.com/puisi-bahasa-sunda
1,c51ddd60-adc7-4b95-b09d-3c4865ba2aaf,KANYERI,eweuh deui seri nu lewih nyeri tibatan di héna...,2023-07-24,94,https://basasunda.com/puisi-bahasa-sunda
2,cb4eb9c6-bf7f-4edc-b997-73a02780e60d,KARUNGING LAIN WAYAH,wanci gayuh ka peuting poék tungkeb haté lain ...,2023-07-24,50,https://basasunda.com/puisi-bahasa-sunda
3,af9fc2c2-0230-4784-8104-cf9271ccfccc,KUDU DAÉK GAWÉ,ieu awak asa lalungsé di rasa asa carapé meure...,2023-07-24,53,https://basasunda.com/puisi-bahasa-sunda
4,2ae6c860-37c6-47f0-aefc-8b74f96b623b,PUISI PANGGGEUING DIRI,naha anjeun téh poho yén mot téh dodoho datang...,2023-07-24,77,https://basasunda.com/puisi-bahasa-sunda


## Open AI Batch Job

In [3]:
client = OpenAI()

In [11]:
def download_job(id: str, file_path: str):
    batch = client.batches.retrieve(id)
    print(batch)

    if (batch.status == "completed"):
        file_content = client.files.content(batch.output_file_id).content
        with open(file_path, "wb") as f:
            f.write(file_content)

## Generate BEIR Dataset

In [4]:
class BEIRQueryItem(BaseModel):
    search_term: str
    answer: str


class BEIRQuery(BaseModel):
    queries: list[BEIRQueryItem]

In [ ]:
triplet_job_path = Path("../data/beir/beir_batch_result.jsonl")
download_job("batch_67e64f6818d48190b7572af05d71fa35", triplet_job_path)

Batch(
    id='batch_67e64f6818d48190b7572af05d71fa35',
    completion_window='24h',
    created_at=1743146856,
    endpoint='/v1/chat/completions',
    input_file_id='file-Uveu2sbk5LjLCfpczKzpve',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1743147405,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1743233256,
    failed_at=None,
    finalizing_at=1743147309,
    in_progress_at=1743146857,
    metadata=None,
    output_file_id='file-T778pxrPkzQDU96zcGj3y7',
    request_counts=BatchRequestCounts(completed=1499, failed=0, total=1499)
)

In [15]:
# load BEIR data mapping
beir_corpus_map = {}
with open(f"../data/beir/beir_map.jsonl", "r") as map_file:
    for line in map_file:
        parsed = json.loads(line)
        beir_corpus_map[parsed["custom_id"]] = parsed["doc_id"]

In [14]:
# generate BEIR corpus file
with open(f"../data/beir/corpus.jsonl", "w") as f:
    for row in df_corpus.itertuples():
        data = {
            "_id": row.id,
            "title": row.title,
            "text": row.content,
        }

        json.dump(data, f)
        f.write("\n")

In [ ]:
beir_total_tokens = 0

with (
    open(triplet_job_path, "r") as batch_file,
    open(f"../data/beir/queries.jsonl", "w") as queries_file,
    open(f"../data/beir/qrels.tsv", "w") as qrels_file,
):
    # write BEIR qrels header
    qrels_file.write("query-id\tcorpus-id\tscore\n")

    # process each batch
    for line in batch_file:
        parsed = json.loads(line)
        model = BEIRQuery(**json.loads(parsed["response"]["body"]["choices"][0]["message"]["content"]))
        beir_total_tokens += parsed["response"]["body"]["usage"]["total_tokens"]

        doc_id = beir_corpus_map[parsed["custom_id"]]
        for item in model.queries:
            query_id = str(uuid4())

            qrels_file.write(f"{query_id}\t{doc_id}\t1\n")

            json.dump({"_id": query_id, "text": item.search_term}, queries_file)
            queries_file.write("\n")

In [18]:
print(f"Total tokens: {beir_total_tokens}")

Total tokens: 1088144

In [20]:
# validate BEIR dataset
corpus, queries, qrels = GenericDataLoader(
    data_folder="../data/beir", 
    qrels_file=f"../data/beir/qrels.tsv"
).load_custom()

  0%|          | 0/1499 [00:00<?, ?it/s]

100%|██████████| 1499/1499 [00:00<00:00, 126037.64it/s]


## Generate Triplet

In [23]:
class TripletItem(BaseModel):
    query: str
    positive_passage: str
    negative_passage: str


class TripetData(BaseModel):
    triplets: list[TripletItem]

In [21]:
triplet_job_path = Path("../data/triplet/triplet_batch_result.jsonl")
download_job("batch_67e64f8db3988190a7a04356370a1af5", triplet_job_path)

Batch(
    id='batch_67e64f8db3988190a7a04356370a1af5',
    completion_window='24h',
    created_at=1743146893,
    endpoint='/v1/chat/completions',
    input_file_id='file-KzHK8pZe8AggMc7ARmqRFy',
    object='batch',
    status='completed',
    cancelled_at=None,
    cancelling_at=None,
    completed_at=1743147862,
    error_file_id=None,
    errors=None,
    expired_at=None,
    expires_at=1743233293,
    failed_at=None,
    finalizing_at=1743147751,
    in_progress_at=1743146895,
    metadata=None,
    output_file_id='file-MSzVH1XW3K5RsnvqLdKXeT',
    request_counts=BatchRequestCounts(completed=1499, failed=0, total=1499)
)

In [24]:
marco_total_tokens = 0

with (
    open(triplet_job_path, "r") as batch_file,
    open(f"../data/triplet/triplet.jsonl", "w") as triplet_file,
):
    for line in batch_file:
        parsed = json.loads(line)
        model = TripetData(**json.loads(parsed["response"]["body"]["choices"][0]["message"]["content"]))
        marco_total_tokens += parsed["response"]["body"]["usage"]["total_tokens"]

        for item in model.triplets:
            data = {
                "query": item.query,
                "positive": item.positive_passage,
                "negative": item.negative_passage,
            }

            json.dump(data, triplet_file)
            triplet_file.write("\n")

In [25]:
print(f"Total tokens: {marco_total_tokens}")

Total tokens: 1510183